In [5]:
# Notebook 01: Merge and Feature Build
#
# Purpose:
# - Load cleaned Alabama_SoS and USDA county-level datasets
# - Inspect columns and merge keys
# - Merge datasets into one county-level modeling table
# - Create derived features for PCA
# - Export the merged dataset

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

In [7]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data" / "cleaned"

AL_SOS_DIR = DATA_DIR / "Alabama_SoS"
USDA_DIR = DATA_DIR / "USDA"

ballots_file = AL_SOS_DIR / "2024 General Election Total Ballots Cast.csv"
race_file = AL_SOS_DIR / "2024 General Election Participation by Race.csv"
age_file = AL_SOS_DIR / "2024 General Election Participation by Age.csv"
gender_file = AL_SOS_DIR / "2024 General Election Participation by Gender.csv"

education_file = USDA_DIR / "Educational attainment for adults age 25 and older for the United States, States, and counties, 1970–2023.csv"
population_file = USDA_DIR / "Population estimates for the United States, States, and counties, 2020–23.csv"
poverty_file = USDA_DIR / "Poverty estimates for the United States, States, and counties, 2023.csv"
income_file = USDA_DIR / "Unemployment and median household income for the United States, States, and counties, 2000–23.csv"

In [8]:
ballots = pd.read_csv(ballots_file)
race = pd.read_csv(race_file)
age = pd.read_csv(age_file)
gender = pd.read_csv(gender_file)

education = pd.read_csv(education_file)
population = pd.read_csv(population_file)
poverty = pd.read_csv(poverty_file)
income = pd.read_csv(income_file)

In [9]:
datasets = {
    "ballots": ballots,
    "race": race,
    "age": age,
    "gender": gender,
    "education": education,
    "population": population,
    "poverty": poverty,
    "income": income
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

ballots: (25, 8)
race: (25, 11)
age: (25, 11)
gender: (25, 5)
education: (3432, 5)
population: (4290, 5)
poverty: (1650, 5)
income: (0, 5)


In [10]:
for name, df in datasets.items():
    print(f"\n{name.upper()} COLUMNS")
    print(df.columns.tolist())


BALLOTS COLUMNS
['County', 'Registered Voters', 'Total Ballots Cast', 'Democrat Ballots', 'Republican Ballots ', 'Absentee Ballots', 'Turnout as a Percentage of Registered Voters', 'Absentee as Percentage of Total Ballots']

RACE COLUMNS
['County', 'Total Ballots', 'Total Asian', 'Total American Indian', 'Total Black', 'Total Federally Registered', 'Total Hispanic', 'Total Korean', 'Total Other', 'Total White', 'Total Not Identified']

AGE COLUMNS
['County', 'Total Ballots Cast', 'Age 18 - 29', 'Age 30 -39', 'Age 40 - 49', 'Age 50 - 59', 'Age 60 - 69', 'Age 70 - 79', 'Age 80 - 89', 'Age 90 -99', 'Age 100+']

GENDER COLUMNS
['County', 'Total Ballots Cast', 'Total Femal', 'Total Male', 'Total Not Identified']

EDUCATION COLUMNS
['FIPS Code', 'State', 'Area name', 'Attribute', 'Value']

POPULATION COLUMNS
['FIPStxt', 'State', 'Area_Name', 'Attribute', 'Value']

POVERTY COLUMNS
['FIPS_Code', 'State', 'Area_Name', 'Attribute', 'Value']

INCOME COLUMNS
['FIPS_Code', 'State', 'Area_Name', 'A

In [11]:
def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^\w_]", "", regex=True)
    )
    return df

ballots = clean_column_names(ballots)
race = clean_column_names(race)
age = clean_column_names(age)
gender = clean_column_names(gender)
education = clean_column_names(education)
population = clean_column_names(population)
poverty = clean_column_names(poverty)
income = clean_column_names(income)

In [12]:
for name, df in {
    "ballots": ballots,
    "race": race,
    "age": age,
    "gender": gender,
    "education": education,
    "population": population,
    "poverty": poverty,
    "income": income
}.items():
    print(f"\n{name.upper()} COLUMNS")
    print(df.columns.tolist())


BALLOTS COLUMNS
['county', 'registered_voters', 'total_ballots_cast', 'democrat_ballots', 'republican_ballots', 'absentee_ballots', 'turnout_as_a_percentage_of_registered_voters', 'absentee_as_percentage_of_total_ballots']

RACE COLUMNS
['county', 'total_ballots', 'total_asian', 'total_american_indian', 'total_black', 'total_federally_registered', 'total_hispanic', 'total_korean', 'total_other', 'total_white', 'total_not_identified']

AGE COLUMNS
['county', 'total_ballots_cast', 'age_18__29', 'age_30_39', 'age_40__49', 'age_50__59', 'age_60__69', 'age_70__79', 'age_80__89', 'age_90_99', 'age_100']

GENDER COLUMNS
['county', 'total_ballots_cast', 'total_femal', 'total_male', 'total_not_identified']

EDUCATION COLUMNS
['fips_code', 'state', 'area_name', 'attribute', 'value']

POPULATION COLUMNS
['fipstxt', 'state', 'area_name', 'attribute', 'value']

POVERTY COLUMNS
['fips_code', 'state', 'area_name', 'attribute', 'value']

INCOME COLUMNS
['fips_code', 'state', 'area_name', 'attribute',

In [13]:
def standardize_county(series):
    return (
        series.astype(str)
        .str.upper()
        .str.strip()
        .str.replace(" COUNTY", "", regex=False)
    )

ballots["county"] = standardize_county(ballots["county"])
race["county"] = standardize_county(race["county"])
age["county"] = standardize_county(age["county"])
gender["county"] = standardize_county(gender["county"])

education["county"] = standardize_county(education["area_name"])
population["county"] = standardize_county(population["area_name"])
poverty["county"] = standardize_county(poverty["area_name"])
income["county"] = standardize_county(income["area_name"])

In [15]:
def pivot_usda(df, county_col="county", attr_col="attribute", value_col="value"):
    wide = df.pivot_table(
        index=county_col,
        columns=attr_col,
        values=value_col,
        aggfunc="first"
    ).reset_index()
    wide.columns.name = None
    return wide

education_wide = pivot_usda(education)
population_wide = pivot_usda(population)
poverty_wide = pivot_usda(poverty)

In [16]:
for name, df in {
    "education_wide": education_wide,
    "population_wide": population_wide,
    "poverty_wide": poverty_wide
}.items():
    print(f"\n{name.upper()} COLUMNS")
    print(df.columns.tolist())


EDUCATION_WIDE COLUMNS
['county', '2013 Rural-urban Continuum Code', '2013 Urban Influence Code', '2023 Rural-urban Continuum Code', '2024 Urban Influence Code', "Bachelor's degree or higher, 1990", "Bachelor's degree or higher, 2000", "Bachelor's degree or higher, 2008-12", "Bachelor's degree or higher, 2019-23", 'Four years of college or higher, 1970', 'Four years of college or higher, 1980', 'High school diploma only, 1970', 'High school diploma only, 1980', 'High school graduate (or equivalency), 1990', 'High school graduate (or equivalency), 2000', 'High school graduate (or equivalency), 2008-12', 'High school graduate (or equivalency), 2019-23', 'Less than a high school diploma, 1970', 'Less than a high school diploma, 1980', 'Less than high school graduate, 1990', 'Less than high school graduate, 2000', 'Less than high school graduate, 2008-12', 'Less than high school graduate, 2019-23', 'Percent of adults completing four years of college or higher, 1970', 'Percent of adults co

In [17]:
ballots_sub = ballots[[
    "county",
    "registered_voters",
    "total_ballots_cast",
    "turnout_as_a_percentage_of_registered_voters",
    "absentee_as_percentage_of_total_ballots"
]].copy()

ballots_sub["turnout_rate"] = ballots_sub["turnout_as_a_percentage_of_registered_voters"] / 100
ballots_sub["absentee_rate"] = ballots_sub["absentee_as_percentage_of_total_ballots"] / 100

ballots_sub.head()

,county,registered_voters,total_ballots_cast,turnout_as_a_percentage_of_registered_voters,absentee_as_percentage_of_total_ballots,turnout_rate,absentee_rate
0,BALDWIN,207643,122542,59.02,6.34,0.5902,0.0634
1,BARBOUR,17666,9919,56.15,5.98,0.5615,0.0598
2,BULLOCK,7181,4144,57.71,9.80,0.5771,0.0980
3,BUTLER,14530,8530,58.71,5.44,0.5871,0.0544
4,CHOCTAW,10767,6692,62.15,6.75,0.6215,0.0675


In [18]:
education_sub = education_wide[[
    "county",
    "Percent of adults who are not high school graduates, 2019-23",
    "Percent of adults with a bachelor's degree or higher, 2019-23"
]].copy()

education_sub = education_sub.rename(columns={
    "Percent of adults who are not high school graduates, 2019-23": "pct_less_hs",
    "Percent of adults with a bachelor's degree or higher, 2019-23": "pct_bachelors"
})

education_sub.head()

,county,pct_less_hs,pct_bachelors
0,AUTAUGA,9.721098,28.282680
1,BALDWIN,8.268600,32.797637
2,BARBOUR,22.186295,11.464715
3,BIBB,19.659783,11.468207
4,BLOUNT,17.303798,15.579030


In [19]:
population_sub = population_wide[[
    "county",
    "POP_ESTIMATE_2023"
]].copy()

population_sub = population_sub.rename(columns={
    "POP_ESTIMATE_2023": "population_2023"
})

population_sub.head()

,county,population_2023
0,AUTAUGA,60342.0
1,BALDWIN,253507.0
2,BARBOUR,24585.0
3,BIBB,21868.0
4,BLOUNT,59816.0


In [20]:
poverty_sub = poverty_wide[[
    "county",
    "PCTPOVALL_2023"
]].copy()

poverty_sub = poverty_sub.rename(columns={
    "PCTPOVALL_2023": "poverty_rate"
})

poverty_sub.head()

,county,poverty_rate
0,AUTAUGA,11.7
1,BALDWIN,10.0
2,BARBOUR,25.5
3,BIBB,19.4
4,BLOUNT,12.8


In [21]:
age_sub = age.copy()

age_sub["age_18_29_share"] = age_sub["age_18__29"] / age_sub["total_ballots_cast"]

age_sub = age_sub[[
    "county",
    "age_18_29_share"
]].copy()

age_sub.head()

,county,age_18_29_share
0,BARBOUR,0.096326
1,BULLOCK,0.106456
2,BUTLER,0.098743
3,CHOCTAW,0.113017
4,CRENSHAW,0.112121


In [22]:
merged = ballots_sub.merge(population_sub, on="county", how="left")
merged = merged.merge(poverty_sub, on="county", how="left")
merged = merged.merge(education_sub, on="county", how="left")
merged = merged.merge(age_sub, on="county", how="left")

merged.head()

,county,registered_voters,total_ballots_cast,turnout_as_a_percentage_of_registered_voters,absentee_as_percentage_of_total_ballots,turnout_rate,absentee_rate,population_2023,poverty_rate,pct_less_hs,pct_bachelors,age_18_29_share
0,BALDWIN,207643,122542,59.02,6.34,0.5902,0.0634,253507.0,10.0,8.268600,32.797637,0.096326
1,BARBOUR,17666,9919,56.15,5.98,0.5615,0.0598,24585.0,25.5,22.186295,11.464715,0.096326
2,BULLOCK,7181,4144,57.71,9.80,0.5771,0.0980,9897.0,33.6,25.955544,8.999729,0.106456
3,BUTLER,14530,8530,58.71,5.44,0.5871,0.0544,18382.0,23.6,12.405044,13.764813,0.098743
4,CHOCTAW,10767,6692,62.15,6.75,0.6215,0.0675,12252.0,24.8,15.781285,13.914203,0.113017


In [23]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 12 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   county                                        25 non-null     object 
 1   registered_voters                             25 non-null     int64  
 2   total_ballots_cast                            25 non-null     int64  
 3   turnout_as_a_percentage_of_registered_voters  25 non-null     float64
 4   absentee_as_percentage_of_total_ballots       25 non-null     float64
 5   turnout_rate                                  25 non-null     float64
 6   absentee_rate                                 25 non-null     float64
 7   population_2023                               24 non-null     float64
 8   poverty_rate                                  24 non-null     float64
 9   pct_less_hs                                   24 non-null     float

In [24]:
merged.isna().sum().sort_values(ascending=False)

poverty_rate                                    1
pct_less_hs                                     1
pct_bachelors                                   1
population_2023                                 1
turnout_as_a_percentage_of_registered_voters    0
total_ballots_cast                              0
registered_voters                               0
county                                          0
absentee_rate                                   0
turnout_rate                                    0
absentee_as_percentage_of_total_ballots         0
age_18_29_share                                 0
dtype: int64

In [25]:
merged.describe()

,registered_voters,total_ballots_cast,turnout_as_a_percentage_of_registered_voters,absentee_as_percentage_of_total_ballots,turnout_rate,absentee_rate,population_2023,poverty_rate,pct_less_hs,pct_bachelors,age_18_29_share
count,2.500000e+01,2.500000e+01,25.000000,25.000000,25.000000,25.000000,24.000000,24.000000,24.000000,24.000000,25.000000
mean,1.956647e+05,1.139386e+05,57.553600,7.269600,0.575536,0.072696,54129.958333,24.416667,15.467159,17.476680,0.109385
std,7.675316e+05,4.517448e+05,4.702273,2.509201,0.047023,0.025092,98936.820379,6.375747,4.094101,6.534449,0.013617
min,6.572000e+03,4.056000e+03,44.680000,3.650000,0.446800,0.036500,7341.000000,10.000000,8.268600,8.999729,0.085848
25%,9.820000e+03,6.105000e+03,55.160000,5.440000,0.551600,0.054400,11588.750000,19.075000,12.414252,12.969478,0.098743
50%,1.571000e+04,8.530000e+03,58.850000,6.750000,0.588500,0.067500,18376.000000,23.700000,15.256294,16.488905,0.107438
75%,2.826800e+04,1.500900e+04,61.570000,8.330000,0.615700,0.083300,33894.000000,29.800000,17.557432,19.374789,0.115842
max,3.861929e+06,2.272911e+06,62.650000,14.000000,0.626500,0.140000,411640.000000,33.800000,25.955544,34.274028,0.150505


In [26]:
OUTPUT_DIR = BASE_DIR / "data" / "merged"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

merged.to_csv(OUTPUT_DIR / "gc_cpri_model_input.csv", index=False)